In [0]:
%sql
SELECT
    COUNT(*) AS TotalDates,
    COUNT(DISTINCT DateKey) AS UniqueDateKeys,
    MIN(FullDate) AS EarliestDate,
    MAX(FullDate) AS LatestDate
FROM adb_adventureworks_project.gold.dim_date;

## Generate the simulated CSV in ADLS

In [0]:
import csv
import io

# Use an existing valid sales record as a template
sample = (
    spark.table("adb_adventureworks_project.gold.fact_sales")
    .select("ProductKey", "CustomerKey", "TerritoryKey")
    .limit(1)
    .first()
)

# Create a simulated 2018 sales record
record = {
    "OrderDate": "2018-01-15",
    "StockDate": "2018-01-01",
    "OrderNumber": "SO-SIM-2018-0001",
    "ProductKey": sample["ProductKey"],
    "CustomerKey": sample["CustomerKey"],
    "TerritoryKey": sample["TerritoryKey"],
    "OrderLineItem": 1,
    "OrderQuantity": 2
}

# Generate CSV content
buffer = io.StringIO()
writer = csv.DictWriter(buffer, fieldnames=list(record.keys()))
writer.writeheader()
writer.writerow(record)

# New year folder
file_path = (
    "abfss://bronze@adventureworkadls2.dfs.core.windows.net/"
    "Sales_2018/AdventureWorks_Sales_2018_simulated.csv"
)

dbutils.fs.put(file_path, buffer.getvalue(), overwrite=False)

print("Simulated sales file created:")
print(file_path)


In [0]:
%sql

SELECT
    OrderNumber,
    OrderDate,
    ProductKey,
    CustomerKey,
    TerritoryKey,
    OrderLineItem,
    OrderQuantity
FROM adb_adventureworks_project.silver.sales_silver_table
WHERE OrderNumber = 'SO-SIM-2018-0001';
